# BM25 & probabilistic retrieval — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cos(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)))
print('setup ok')

## Rare terms, saturated term frequency, and length normalization
This notebook scores one query by BM25, then isolates the idf, saturation, length, ranking, and parameter effects that make BM25 more robust than raw counts.

In [ ]:
# 1) BM25 idf: rarer query terms get larger weights
corpus=[['neural','search','search'],['graph','search'],['neural','retrieval','models'],['graph','retrieval','search','search']]
N=len(corpus)
def idf(term):
    df=sum(term in d for d in corpus)
    return math.log(1+(N-df+0.5)/(df+0.5))
vals=[idf('search'),idf('neural')]
plt.figure(figsize=(6,3)); plt.bar(['search df=3','neural df=2'],vals); plt.ylabel('BM25 idf'); plt.title('rarer term gets more weight')
assert abs(vals[0]-0.3566749439)<1e-9 and abs(vals[1]-0.6931471806)<1e-9

In [ ]:
# 2) Term-frequency saturation bends repeated matches
k1=1.2; b=.75; avgdl=3.0; dl=3
mult=[tf*(k1+1)/(tf+k1*(1-b+b*dl/avgdl)) for tf in [1,2,5,10]]
plt.figure(figsize=(6,3)); plt.plot([1,2,5,10],mult,'-o'); plt.xlabel('tf'); plt.ylabel('BM25 tf multiplier'); plt.title('repetition helps, then saturates')
assert abs(mult[0]-1.0)<1e-9 and mult[-1]-mult[-2]<mult[1]-mult[0]

In [ ]:
# 3) Length normalization discounts longer documents with the same tf
lengths=[2,3,4,8]; tf=1
scores=[tf*(k1+1)/(tf+k1*(1-b+b*dl/avgdl)) for dl in lengths]
plt.figure(figsize=(6,3)); plt.plot(lengths,scores,'-o'); plt.xlabel('document length'); plt.ylabel('tf component'); plt.title('same count scores lower in longer docs')
assert scores[0]>scores[-1]

In [ ]:
# 4) Full BM25 ranking for query: neural search
q=['search','neural']; avgdl=sum(map(len,corpus))/len(corpus)
def bm25(doc):
    s=0
    for term in q:
        tf=doc.count(term)
        if tf:
            s += idf(term)*tf*(k1+1)/(tf+k1*(1-b+b*len(doc)/avgdl))
    return s
scores=[bm25(d) for d in corpus]
plt.figure(figsize=(6,3)); plt.bar(['d1','d2','d3','d4'],scores); plt.ylabel('BM25'); plt.title('d1 wins by matching both query terms')
assert np.allclose(scores,[1.1835752285,0.4129920404,0.6931471806,0.4483913581])

In [ ]:
# 5) Turning off length normalization changes the contribution shape
bs=[0,.25,.75,1.0]; doc=corpus[3]
score_b=[]
for bb in bs:
    tf=doc.count('search'); score_b.append(idf('search')*tf*(k1+1)/(tf+k1*(1-bb+bb*len(doc)/avgdl)))
plt.figure(figsize=(6,3)); plt.plot(bs,score_b,'-o'); plt.xlabel('b'); plt.ylabel('search contribution in long d4'); plt.title('larger b penalizes the long document')
assert score_b[0]>score_b[-1]